<a href="https://colab.research.google.com/github/Bidisha314/Study-APP/blob/main/Study_RAG_APP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **How to use Gemini File Search Tool as an easy RAG alternative**
## Guide: [How to Use the File Search Tool in Gemini API for Easy RAG Integration](https://pinggy.io/blog/how_to_use_file_search_tool_in_gemini_api_for_easy_rag_integration/)

Prerequisites:
*   GPU-T4 (for faster result)
*   Python 3.8 or higher Version
*   Create Google Gemini API key from https://aistudio.google.com/api-keys


## Step 1: Create a new folder-template
*It creates folder named *templates* inside the content*

In [ ]:
!rm -rf templates
!mkdir -p templates
print("Templates folder created successfully.")


Templates folder created successfully.


## Step 2: Create UI design HTML file
File name `index.html`

In [ ]:
%%writefile /content/templates/index.html
<!DOCTYPE html>
<html>
<head>
<title>Study RAG</title>
</head>
<body style="font-family: Arial, sans-serif; background-color: #f5f7fa; margin: 0; padding: 0;">
<div style="max-width: 700px; margin: 40px auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); text-align: center;">
<h2 style="color: #2c3e50;">Study RAG</h2>
<p style="color: #555; font-size: 15px;">Upload your notes and ask anything based on them.</p>

<!-- Tips Box -->
<div style="background-color:#e8f4ff; color:#084298; padding:12px; border-radius:6px; font-size:14px; margin:15px 0; text-align:left;">
<strong>Tips:</strong>
<ul style="padding-left:20px; margin:8px 0;">
<li>You can upload study notes in .txt, .pdf or .doc format</li>
<li>Ask clear and specific questions for best answers</li>
<li>Do not repeatedly upload the same file</li>
<li>If you hit limits, try again after some time</li>
</ul>

 </div>
 <form method="POST" enctype="multipart/form-data" style="margin-bottom: 20px;">
 <input type="file" name="file" style="padding: 8px; border: 1px solid #ccc; border-radius: 5px;">
 <button type="submit" style="padding: 8px 18px; border: none; background-color: #007bff; color: white; border-radius: 5px; cursor: pointer;">Upload Notes</button>
 </form>

 {% if file_message %}
 <div style="background-color:#d4edda; color:#155724; padding:10px; border-radius:5px; margin-top:20px; font-size:14px;">✔️ {{ file_message }}</div>
 {% endif %}

<hr>


<form method="POST" style="margin-top: 20px;">
  <input type="text" name="question" placeholder="Ask something..." style="width: 90%; padding: 10px; border: 1px solid #ccc; border-radius: 6px; font-size: 15px;">
  <button type="submit" style="margin-top: 10px; padding: 10px 20px; background-color: #28a745; color: white; border: none; border-radius: 5px; cursor: pointer;">Ask</button>
</form>


{% if answer %}
<div style="text-align: left; margin-top: 15px; line-height: 1.5;">
{{ answer | safe }}
</div>
{% endif %}
</div>
</body>
</html>

Writing /content/templates/index.html


## Step 3: Set up Pinggy Tunnel


#### Install Pinggy:

*   Check : [Pinggy.io](https:/pinggy.io/)
*   Check: [Python SDK](https://pypi.org/project/pinggy/)
*   [How to Use the File Search Tool in Gemini API for Easy RAG Integration](https://pinggy.io/blog/how_to_use_file_search_tool_in_gemini_api_for_easy_rag_integration/)




In [ ]:
!pip install pinggy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 23.1 MB/s eta 0:00:00


#### Create Pinggy Tunnel
It is running on `port:8000` (you can change port by changing `"localhost:port_number"`)

In [ ]:
import pinggy
tunnel1= pinggy.start_tunnel(
    forwardto="localhost:8000"
)
print(f"Tunnel1 started ~ URLs:{tunnel1.urls}")

Tunnel1 started ~ URLs:['http://utjcj-34-16-216-119.a.free.pinggy.link', 'https://utjcj-34-16-216-119.a.free.pinggy.link']


## Step 4: Build the Alternatiev APP to RAG: STUDY RAG
*I used Flask to build the app.*

*   You have to `import genai` from google.  
*   Gemini model used: gemini-2.0-flash-lite-preview
*   Create API Key: https://aistudio.google.com/api-keys





In [ ]:
!pip install markdown
from flask import Flask, render_template, request
from google import genai

import os
import time

# INSERT YOUR OWN KEY HERE
client = genai.Client(api_key="Your_API_Key")

app = Flask(__name__)
file_search_store = None

@app.route('/', methods=['GET', 'POST'])
def index():
    global file_search_store
    answer = None
    file_message = None

    if request.method == 'POST':
        # File upload debugging
        if 'file' in request.files and request.files['file'].filename != "":
            file = request.files['file']
            filepath = "./" + file.filename
            print("📌 Received file:", file.filename)

            file.save(filepath)
            print("📌 Saved to:", filepath)

            try:
                if not file_search_store:
                    print("🛠 Creating File Search Store...")
                    store = client.file_search_stores.create(
                        config={'display_name': 'study-notes-store'}
                    )
                    file_search_store = store.name
                    print("✔ Store created:", file_search_store)

                print("⬆ Uploading file to Gemini File Search...")
                op = client.file_search_stores.upload_to_file_search_store(
                    file=filepath,
                    file_search_store_name=file_search_store
                )

                while not op.done:
                    print("⏳ Indexing in progress...")
                    time.sleep(1)
                    op = client.operations.get(op)  # < FIXED

                print("🎉 File indexing completed!")
                os.remove(filepath)
                file_message = "File uploaded & indexed successfully!"

            except Exception as e:
                print("❌ FILE UPLOAD ERROR:", e)
                answer = "Upload Error: " + str(e)

        # Question Answering
        if 'question' in request.form and file_search_store:
            question = request.form['question']
            print("❓ Question:", question)

            try:
                response = client.models.generate_content(
                    model="gemini-2.0-flash-lite-preview",
                    contents=question,
                  )

                import markdown
                raw_answer = response.text
                formatted_html = markdown.markdown(raw_answer)
                answer = f"""
                 <div style='background:#ffffff; padding:15px; border-radius:8px;
                 border:1px solid #ddd; text-align:left; line-height:1.6;'>
                 {formatted_html}
                 </div>
                 """
                print("🤖 Answer:", raw_answer)

            except Exception as e:
               print("❌ ANSWER ERROR:", e)

               error_message = str(e)

               if "RESOURCE_EXHAUSTED" in error_message or "429" in error_message:
                  answer = (
                            "⚠️ Resource Limit Reached!\n\n"
                            "You’ve hit the daily quota for the AI service.\n"
                            "Please wait a while and try again.\n\n"
                            "Tips:\n"
                            "- Avoid re-uploading the same notes repeatedly\n"
                            "- Ask fewer long questions during testing\n"
                          "- Enable billing in Google Cloud to increase your limits\n"
                           )
               else:
                 answer = "⚠️ Something went wrong. Try again!\n\nDetails: " + error_message
    return render_template('index.html', answer=answer, file_message=file_message)


app.run(host="0.0.0.0", port=8000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Nov/2025 08:03:09] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Nov/2025 08:03:09] "GET /favicon.ico HTTP/1.1" 404 -


📌 Received file: Assignment.docx
📌 Saved to: ./Assignment.docx
🛠 Creating File Search Store...
✔ Store created: fileSearchStores/studynotesstore-dez8occy4tdu
⬆ Uploading file to Gemini File Search...
⏳ Indexing in progress...


INFO:werkzeug:127.0.0.1 - - [21/Nov/2025 08:03:24] "POST / HTTP/1.1" 200 -


🎉 File indexing completed!
❓ Question: Claudius


INFO:werkzeug:127.0.0.1 - - [21/Nov/2025 08:03:32] "POST / HTTP/1.1" 200 -


🤖 Answer: Claudius refers to several historical figures, but the most well-known is:

**Claudius (Tiberius Claudius Caesar Augustus Germanicus)** (10 BC – 54 AD)

*   He was the **fourth Roman Emperor**, ruling from 41 AD to 54 AD.
*   He was a member of the Julio-Claudian dynasty.
*   He was considered an unlikely emperor due to perceived physical and social disabilities.
*   He made significant contributions to the Roman Empire, including:
    *   Conquest of Britain (43 AD)
    *   Building of infrastructure, such as harbors and aqueducts.
    *   Expansion of Roman citizenship.
    *   Administrative reforms.
*   He was likely poisoned, possibly by his wife Agrippina the Younger, to make way for her son Nero to become emperor.

Do you have any specific questions about Claudius? I can also tell you about:

*   His family and relationships
*   His life before becoming emperor
*   His reign and accomplishments
*   His death and legacy
*   How he is depicted in history and literature (

**Sample Text for Testing:**

I, Claudius is a historical novel by English writer Robert Graves, published in 1934.
Written in the form of an autobiography of the Roman Emperor Claudius, it tells the
history of the Julio-Claudian dynasty and the Roman Empire from Julius Caesar's
assassination in 44 BC to Caligula's assassination in AD 41.

The book is written as a first-person narrative of the life of Roman Emperor Claudius.
Graves portrays Claudius as a sympathetic character rather than the bumbling idiot that
he is often depicted as in history.

*You can make .txt or .pdf or .doc file to upload*

**Question Sample:**

Search "Claudius"